Exam Task:
1. Load the Pima Indians Diabetes dataset and handle zero-value anomalies in
features like Glucose or BloodPressure
2. Write a custom bootstrap function using np.random.choice (sampling with
replacement) to train an ensemble of to unpruned𝐵 = 10 20
DecisionTreeClassifier base learners
3. Implement a custom aggregation function that applies majority voting to make
final predictions on test samples.
4. Compare the test set Accuracy and Confusion Matrix of your custom Bagging
model against a standalone single Decision Tree

### Step 1: Load and Clean the Pima Indians Diabetes Dataset
We will download the dataset, split it into train/test sets, and impute zero-value anomalies using the training set medians to avoid data leakage.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Load dataset directly from URL
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
column_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=column_names)

# Split into features (X) and target (y)
X = df.drop(columns='Outcome')
y = df['Outcome']

# Split into Train and Test sets first to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Zero-value anomalies handling (for specific columns where 0 is biologically invalid)
zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Impute zeros with training set medians
for col in zero_columns:
    median_val = X_train[col].replace(0, np.nan).median()
    X_train[col] = X_train[col].replace(0, median_val)
    X_test[col] = X_test[col].replace(0, median_val)

# Convert to numpy arrays for the bootstrap and custom classifier functions
X_train_arr, y_train_arr = X_train.to_numpy(), y_train.to_numpy()
X_test_arr, y_test_arr = X_test.to_numpy(), y_test.to_numpy()

print("Data loaded and cleaned successfully.")
print(f"Train shape: {X_train_arr.shape}, Test shape: {X_test_arr.shape}")

Data loaded and cleaned successfully.
Train shape: (614, 8), Test shape: (154, 8)


### Steps 2 & 3: Custom Bootstrap Function, Ensemble Training, and Majority Voting Aggregation

In [ ]:
def custom_bootstrap_bagging(X, y, B=20):
    """
    Trains an ensemble of B unpruned DecisionTreeClassifiers
    using custom bootstrap sampling.
    """
    n_samples = X.shape[0]
    models = []

    # Set random seed for reproducibility
    np.random.seed(42)

    for i in range(B):
        # Custom bootstrap using np.random.choice with replacement
        bootstrap_indices = np.random.choice(n_samples, size=n_samples, replace=True)
        X_bootstrap = X[bootstrap_indices]
        y_bootstrap = y[bootstrap_indices]

        # Train unpruned base learner
        clf = DecisionTreeClassifier(random_state=42 + i)
        clf.fit(X_bootstrap, y_bootstrap)
        models.append(clf)

    return models

def custom_predict_bagging(models, X):
    """
    Predicts class labels using majority voting across models.
    """
    # Get predictions from each base model
    preds = np.array([model.predict(X) for model in models]) # Shape: (B, n_samples)

    # Majority voting aggregation
    # Transpose to (n_samples, B) and find the mode
    final_predictions = []
    for sample_preds in preds.T:
        # Compute the most common prediction (majority vote)
        vote = np.bincount(sample_preds).argmax()
        final_predictions.append(vote)

    return np.array(final_predictions)

# Train the ensemble with B = 20 models
B = 20
ensemble_models = custom_bootstrap_bagging(X_train_arr, y_train_arr, B=B)
ensemble_preds = custom_predict_bagging(ensemble_models, X_test_arr)

print(f"Trained an ensemble of B={B} unpruned Decision Trees.")

Trained an ensemble of B=20 unpruned Decision Trees.


### Step 4: Compare Against a Standalone Single Decision Tree

In [ ]:
# Train a single standalone unpruned Decision Tree
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train_arr, y_train_arr)
single_tree_preds = single_tree.predict(X_test_arr)

# Evaluate accuracy
acc_bagging = accuracy_score(y_test_arr, ensemble_preds)
acc_single = accuracy_score(y_test_arr, single_tree_preds)

# Evaluate confusion matrices
cm_bagging = confusion_matrix(y_test_arr, ensemble_preds)
cm_single = confusion_matrix(y_test_arr, single_tree_preds)

# Display Results
print("=================== COMPARISON RESULTS ===================\n")
print(f"Custom Bagging Ensemble (B={B}) Accuracy : {acc_bagging:.4f}")
print(f"Single Decision Tree Accuracy          : {acc_single:.4f}\n")

print("Custom Bagging Ensemble Confusion Matrix:")
print(cm_bagging)
print("\nSingle Decision Tree Confusion Matrix:")
print(cm_single)

=================== COMPARISON RESULTS ===================

Custom Bagging Ensemble (B=20) Accuracy : 0.7597
Single Decision Tree Accuracy          : 0.6818

Custom Bagging Ensemble Confusion Matrix:
[[87 13]
 [24 30]]

Single Decision Tree Confusion Matrix:
[[79 21]
 [28 26]]
